In [ ]:
import torch
from torch import nn
from torch.utils.data import random_split, DataLoader, Subset
from torchvision import datasets
from torchvision.transforms import ToTensor
import numpy as np
import os
import pandas as pd
from torch.utils.data import Dataset
# import sklearn
from sklearn.model_selection import train_test_split
from torchvision.transforms import ToTensor, Lambda
import matplotlib.colors as mcolors

from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold


import matplotlib.pyplot as plt
import wandb

import os
import datetime

In [ ]:
# Open questions

# How can I place a hard constrain that Mtot,f<= Mtot, i and M1f<=Mtot,f
# What is the best way to evaluate model perfomance? The absolute error is based on the ratio of masses. 
# Do I have to startify the data?
# I need to do parameter search: lr, epoch 


In [ ]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

In [ ]:
print(device)

In [ ]:
#This is in place to fix all variables so I can gauge how effective changes are 
torch.manual_seed(42)
np.random.seed(42)

# Make training deterministic
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


#### **Put data into a database**

In [ ]:

class CustomDataset(Dataset):
    def __init__(self, labels, data, dir, transform=None, target_transform=None):
        self.labels = labels
        self.data = data
        self.dir = dir
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        
        data = self.data[idx]
        label = self.labels[idx]

        data = torch.from_numpy(data).type(torch.float)
        label = torch.tensor(label)

        if self.transform:
            data = self.transform(data)
        if self.target_transform:
            label = self.target_transform(label)
                            
        return data, label

In [ ]:
#--- Create the dataset ---# 

# 0:Met[Zsun],1:Age[Gyr],2:Rp[Rsun],3:Vinf[km/s],4:Mass1_i[MSUN],5:Mass2_i[Msun],6:R1[Rsun],7:R2[Rsun],8:Label,9:Mass1_f[MSUN],10:Mass2_f[MSUN],11:Sigma
# data_dir = '/projects/b1091/SPH_ML_grid/machine_learning/data_n10k_splot19f.csv'
# data_dir = '/projects/b1091/SPH_ML_grid/machine_learning/data_n10k_splot22f.csv'
data_dir = '../../data_n10k_splot22f.csv'

data_loaded  = np.loadtxt(data_dir, delimiter=',', dtype=float, skiprows = 1)

# Selecting only the columns I want to have the format: 
# 0:Met[Zsun],1:Age[Gyr],2:Rp[Rsun],3:Vinf[km/s],4:Mass1_i[MSUN],5:Mass2_i[Msun],6:R1[Rsun],7:R2[Rsun],8:Label,9:Mass1_f[MSUN],10:Mass2_f[MSUN],11:Sigma
data = data_loaded[:, [0, 1, 2, 3, 4, 5, 8]]
initial_masses = data_loaded[:, [4, 5]]
final_masses = data_loaded[:, [9, 10]]

#Re-assing final masses for merger cases to be all in mass1, otherwise there's a lot of confusinon 
merger_mask = data_loaded[:, 8] == 1
swap_rows = merger_mask & (final_masses[:, 0] == 0) & (final_masses[:, 1] > 0)
final_masses[swap_rows, 0], final_masses[swap_rows, 1] = (final_masses[swap_rows, 1], final_masses[swap_rows, 0])

# Now log the masses 
y_data_reg = final_masses
y_data_reg = np.log1p(y_data_reg) / np.log(10)

print(np.min(y_data_reg[:,0]), np.min(y_data_reg[:,1]))
print(np.max(y_data_reg[:,0]), np.max(y_data_reg[:,1]))


#Feature transform: log the x-data
x_data = data[:, 1:-1] # don't need met or labels
x_data[:, 0] = np.log10(x_data[:, 0]) # log ages
x_data[:, 2] = np.log10(x_data[:, 2]) # log vinfs
x_data[:, 3] = np.log1p(x_data[:, 3]) / np.log(10)
x_data[:, 4] = np.log1p(x_data[:, 4]) / np.log(10)
print(np.shape(x_data), np.shape(y_data_reg))
print(np.min(x_data[:, 3]), np.max(x_data[:, 3]))
#Save the original input for later 
x_data_og = x_data

#Split into Training + Temporary (Validation + Test)
X_train, X_temp, y_train, y_temp = train_test_split(x_data, y_data_reg, test_size=0.30,  random_state=42)

# Split the temporary into validation and testing sets 
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(np.shape(X_train), np.shape(X_val), np.shape(X_test))

# Standard Normalize the data 
train_mean = X_train.mean(axis=0)
train_std = X_train.std(axis=0)

y_train_mean = y_train.mean(axis=0)
y_train_std = y_train.std(axis=0)

X_train = (X_train - train_mean) / train_std
X_test = (X_test - train_mean) / train_std  # apply train stats
X_val = (X_val - train_mean) / train_std  # apply train stats

y_train = (y_train - y_train_mean) / y_train_std
y_test = (y_test - y_train_mean) / y_train_std  # apply train stats
y_val = (y_val - y_train_mean) / y_train_std  # apply train stats

# Create separate datasets for train and test
train_dataset = CustomDataset(dir=data_dir, labels = y_train, data = X_train, transform=None, target_transform = None)
val_dataset = CustomDataset(dir=data_dir, labels = y_val, data = X_val, transform=None, target_transform = None)
test_dataset = CustomDataset(dir=data_dir, labels = y_test, data = X_test, transform=None, target_transform = None)

#Take this out for final iteration, it is here to make sure data is shuffled the same each time
g = torch.Generator()
g.manual_seed(42)

train_dataloader = DataLoader(train_dataset, batch_size=512, shuffle=True, generator=g)
val_dataloader = DataLoader(val_dataset, batch_size=128, shuffle=False, generator=g)
test_dataloader = DataLoader(test_dataset, batch_size=128, shuffle=False, generator=g)


for batch_data, (batch_labels) in train_dataloader:

    print("Batch data shape:", batch_data.shape)   # Correct way to check shape
    print("Batch labels shape:", batch_labels.shape)
    print("Labels dtype:", batch_labels.dtype)


### **Build Neural Network**

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        # Shared layers
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(5, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 2),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [ ]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)

    model = model.to(device)
    model.train()

    train_loss, mean_absolute_error = 0, 0
    y_true = [] # Store labels for balanced accuracy
    y_pred = []  # Store labels for balanced accuracy

    for X, y in dataloader:
        X = X.to(device, dtype=torch.float32)
        y = y.to(device, dtype=torch.float32)

        # Get initial (unnormalized masses)
        mass1i = X[:, 3] * train_std[3] + train_mean[3]
        mass2i = X[:, 4] * train_std[4] + train_mean[4]

        # Compute prediction error
        pred = model(X)

        # Calculate overall loss
        loss = loss_fn(pred, y)

        # pred_sum = torch.expm1((pred[:, 0] * y_train_std[0] + y_train_mean[0])* torch.log(torch.tensor(10.0, device=X.device))) + torch.expm1((pred[:, 1]*  y_train_std[1] + y_train_mean[1])* torch.log(torch.tensor(10.0, device=X.device)))
       
        pred_sum = (pred[:, 0] * y_train_std[0] + y_train_mean[0]) + (pred[:, 1]*  y_train_std[1] + y_train_mean[1])

        mass1i = X[:, 3] 
        mass2i = X[:, 4] 
        pred_sum = (pred[:, 0] + pred[:, 1])
        
        penalty = torch.mean(torch.relu(pred_sum - mass1i - mass2i) ** 2)
        
        print("MSE loss", loss.item())
        print("penalty ", (100* penalty).item() )
        train_loss += loss.item() +  100*penalty

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Store predictions and true labels
        y_true.append(y.detach().cpu().numpy())  # Convert tensors to NumPy
        y_pred.append(pred.detach().cpu().numpy())

    train_loss /= num_batches
    
    # stack into (N, 2) arrays
    y_true = np.vstack(y_true)
    y_pred = np.vstack(y_pred)

    # assume y_true and y_pred are torch tensors
    errors = np.abs(y_pred - y_true) 

    mean_absolute_error = np.mean(errors, axis = 0) 

    # Log metrics to wandb.
    run.log({"train_loss": train_loss, "train_abs_error_M1f": mean_absolute_error[0], "train_abs_error_M2f": mean_absolute_error[1]})

def test(dataloader, model, loss_fn, train_mean, train_std, y_train_mean, y_train_std):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model = model.to(device)
    model.eval()
    val_loss, mean_absolute_error = 0, 0
    y_true = []
    y_pred = []
    initial_total_masses = []

    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device, dtype=torch.float32)
            y = y.to(device, dtype=torch.float32)

            pred = model(X)

            # mass1i = X[:, 3] * train_std[3] + train_mean[3]
            # mass2i = X[:, 4] * train_std[4] + train_mean[4]
            # pred_sum = torch.expm1((pred[:, 0] * y_train_std[0] + y_train_mean[0])* torch.log(torch.tensor(10.0, device=X.device))) + torch.expm1((pred[:, 1]*  y_train_std[1] + y_train_mean[1])* torch.log(torch.tensor(10.0, device=X.device)))
            
            mass1i = X[:, 3] * train_std[3] + train_mean[3]
            mass2i = X[:, 4] * train_std[4] + train_mean[4]
            pred_sum = (pred[:, 0] * y_train_std[0] + y_train_mean[0]) + (pred[:, 1]*  y_train_std[1] + y_train_mean[1])
            
            penalty = torch.mean(torch.relu(pred_sum - mass1i - mass2i) ** 2)
            print("Val: MSE loss", loss_fn(pred, y).item())
            print("Val:penalty ", (100*penalty).item())
    
            val_loss += loss_fn(pred, y).item()  + 100* penalty    

            # Store predictions and true labels
            y_true.append(y.detach().cpu().numpy())  # Convert tensors to NumPy
            y_pred.append(pred.detach().cpu().numpy())

            # # Get initial (unnormalized masses)
            # mass1i = X[:, 3].detach().cpu().numpy() * train_std[3] + train_mean[3]
            # mass2i = X[:, 4].detach().cpu().numpy() * train_std[4] + train_mean[4]

            #Elena: changing his 

            mass1i = np.expm1((X[:, 3].detach().cpu().numpy() * train_std[3] + train_mean[3] )* np.log(10))
            mass2i = np.expm1((X[:, 4].detach().cpu().numpy() * train_std[4] + train_mean[4])* np.log(10))

            initial_total_masses.append(mass1i + mass2i) # Msun


    val_loss /= num_batches

    #-- Error metric 1: Absolute Errors in the final masses
    y_true = np.vstack(y_true) # stack into (N, 2) arrays
    y_pred = np.vstack(y_pred)  
    initial_total_masses = np.concatenate(initial_total_masses)
    print(np.min(initial_total_masses), np.max(initial_total_masses))

    mean_absolute_error = np.mean(np.abs(y_pred - y_true) , axis = 0)

    true_mass1 = np.expm1((y_true[:,0] * y_train_std[0] + y_train_mean[0]) * np.log(10))
    true_mass2 = np.expm1((y_true[:,1] * y_train_std[1] + y_train_mean[1]) * np.log(10))

    #-- Error metric 2: Absolute Errors in Mass 1 and Mass 2  
    predicted_mass1 = np.expm1((y_pred[:,0] * y_train_std[0] + y_train_mean[0]) * np.log(10))
    predicted_mass2 = np.expm1((y_pred[:,1] * y_train_std[1] + y_train_mean[1]) * np.log(10))

    # Replace very small numbers by 0
    true_mass1[true_mass1 < 0.001] = 0
    true_mass2[true_mass2 < 0.001] = 0
    predicted_mass1[predicted_mass1 < 0.001] = 0 
    predicted_mass2[predicted_mass2 < 0.001] = 0 

    mean_abs_error_m1 = np.mean(np.abs(predicted_mass1 - true_mass1)) 
    mean_abs_error_m2 = np.mean(np.abs(predicted_mass2 - true_mass2)) 

    #-- Error metric 3: Relative errors for cases where at least one star survives
    mean_rel_error_m1 = np.mean(np.abs(predicted_mass1[true_mass1 != 0.] - true_mass1[true_mass1 != 0. ])/ true_mass1[true_mass1 != 0.])
    mean_rel_error_m2 = np.mean(np.abs(predicted_mass2[true_mass2 != 0.] - true_mass2[true_mass2 != 0. ])/ true_mass2[true_mass2 != 0.])


    #-- Add a check 
    Mtot_final = predicted_mass1 + predicted_mass2
    print("Number of times Mtot,f > Mtot, i ", np.where(Mtot_final > initial_total_masses + 0.1)[0].size)
    print("Examples ", Mtot_final[np.where(Mtot_final > initial_total_masses + 0.1)], initial_total_masses[np.where(Mtot_final > initial_total_masses + 0.1)],)
    print("Number of times M1,f + M2,f > Mtot, f ", np.where(true_mass1 + true_mass2 > Mtot_final + 0.1)[0].size)
    return val_loss, mean_absolute_error, mean_abs_error_m1, mean_abs_error_m2, mean_rel_error_m1, mean_rel_error_m2, Mtot_final, initial_total_masses
    # return val_loss, mean_absolute_error, mean_abs_error_m1, mean_abs_error_m2, mean_rel_error_m1, mean_rel_error_m2


In [ ]:
# Get today's date as string
date_str = datetime.datetime.now().strftime("%m%d")

# Safety stop in Jupyter
wandbname = input("Name for run: ")

if len(wandbname) == 0:
    raise RuntimeError("Stopped — you didn’t provide name for run.")
    

# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="elena-gonzalez-northwestern-university",
    # Set the wandb project where this run will be logged.
    project= "ML_SPH_NN",
    name= wandbname,
    # Track hyperparameters and run metadata.
    config={
        "learning_rate": 'cosine_annealing',
        "architecture": "Regression_NN",
        "dataset": "data_n10k_splot22f.csv",
        "epochs": 1000,
        "width": 521_256_128, 
        "layers": 3,
        "batch_size": 512,
        "labels":2
    },
)

run_id = wandb.run.id  
model_name = f"model_{date_str}_{run_id}"
print(model_name)

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.utils.class_weight import compute_class_weight

# Set Hyper-parameters 
epochs = 1000

# Initialize list for storing validation loss for plotting
val_loss_history = []  # Store per-fold validation losses
lrs = []  # Store per-fold validation learning rates

best_val_score = np.inf
best_model_state = None

# Load the NN 
model = NeuralNetwork()
model = model.float()

# Set loss functions, optimizer, and learning rate scheduler
loss_fn = nn.MSELoss() 
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
scheduler = CosineAnnealingLR(optimizer, T_max=epochs) 

# Begin the training 
for t in range(epochs):
    print(t)
    train(train_dataloader, model, loss_fn, optimizer)
    val_loss, val_mean_absolute_error, val_mean_abs_error_m1, val_mean_abs_error_m2, val_mean_rel_error_m1, val_mean_rel_error_m2 = test(val_dataloader, model, loss_fn, train_mean, train_std, y_train_mean, y_train_std)

    # Compute different accuracies for validation set
    size = len(val_dataloader.dataset)
    num_batches = len(val_dataloader)

    # Log metrics to wandb.
    run.log({"val_loss": val_loss, "val_abs_error_M1f": val_mean_absolute_error[0],"val_abs_error_M2f": val_mean_absolute_error[1], 
             "val_abs_error_m1":val_mean_abs_error_m1, "val_abs_error_m2":val_mean_abs_error_m2, "val_mean_rel_error_m1": val_mean_rel_error_m1, "val_mean_rel_error_m2":val_mean_rel_error_m2})

    if val_loss < best_val_score:
        best_val_score = val_loss
        best_model_state = model.state_dict()

    scheduler.step()  # update the learning rate
    lrs.append(optimizer.param_groups[0]['lr'])


print("Done!")

# Now save the best model as well as the data's mean and standard deviation
checkpoint = {
    "model_state_dict": best_model_state,
    "train_mean": train_mean,
    "train_std": train_std,
    "ytrain_mean": y_train_mean,
    "ytrain_std" : y_train_std,
}
torch.save(checkpoint, model_name)

run.finish()


### Evaluation Metrics

In [ ]:
model = NeuralNetwork()
model_name = model_name
print("Looking at ", model_name)
checkpoint = torch.load(model_name, map_location=torch.device('mps'))
model.load_state_dict(checkpoint["model_state_dict"])

test_loss, test_mean_absolute_error, test_mean_abs_error_m1, test_mean_abs_error_m2, test_mean_rel_error_m1, test_mean_rel_error_m2, mtot_final, mtot_initial= test(test_dataloader, model, loss_fn, train_mean, train_std, y_train_mean, y_train_std)

print(f"\n Test Results:")
print(f"Overall Test Loss: {test_loss:.4f}")
print(f"Absolute Errors M1 [Msun]: {test_mean_abs_error_m1:.4f}")
print(f"Absolute Errors M2 [Msun]: {test_mean_abs_error_m2:.4f}")
print(f"Relative Errors M1,f  : {test_mean_rel_error_m1:.4f}")
print(f"Relative Errors M2,f: {test_mean_rel_error_m2:.4f}")

In [ ]:
plt.scatter(mtot_initial, mtot_final, s = 5)
x = np.linspace(0, int(max(np.max(mtot_final), np.max(mtot_initial))), 100)
plt.plot(x, x, color = 'red')
plt.ylabel('Mtot_final')
plt.xlabel('Mtot_initial')

In [ ]:


def plot_4d_regression(model, X_train, y_train, X_test, y_test, train_mean, train_std,y_train_mean, y_train_std, 
                       feature_idx=(0, 1), fixed_values={}, labels=[] ):
    """
    Plots regression outputs for a PyTorch model using a 2D slice of a higher-dimensional space.
    Produces one panel per output dimension (color gradient).
    
    Parameters:
    - model: Trained PyTorch model.
    - X_train, y_train, X_test, y_test: datasets.
    - train_mean, train_std: normalization stats.
    - feature_idx: Tuple (i, j) specifying which two features to plot.
    - fixed_values: {feature_index: value} for fixing other dimensions.
    - labels: list of strings for axis and titles. Expected: [x_label, y_label, ..., etc].
    """

    # Step 1: Normalize fixed values
    fixed_values_norm = {k: (v - train_mean[k]) / train_std[k] for k, v in fixed_values.items()}

    
    train_mask = np.logical_and.reduce([np.isclose(X_train[:, k], v, rtol=0.01) for k, v in fixed_values_norm.items()])
    test_mask = np.logical_and.reduce([np.isclose(X_test[:, k], v, rtol=0.01) for k, v in fixed_values_norm.items()])

    if (train_mask.sum() == 0 and test_mask.sum() == 0):
        return "No data points!"

    X_train_filtered, y_train_filtered = X_train[train_mask], y_train[train_mask]
    X_test_filtered, y_test_filtered = X_test[test_mask], y_test[test_mask]

    # Step 2: Create meshgrid
    X_train_filtered_unnorm = np.array(X_train_filtered) * train_std + train_mean
    X_test_filtered_unnorm = np.array(X_test_filtered) * train_std + train_mean

    y_train_filtered_unnorm = np.array(y_train_filtered) * y_train_std + y_train_mean # [Mtot,f/Mtot,i, M1,f/Mtot,f]
    y_test_filtered_unnorm = np.array(y_test_filtered) * y_train_std + y_train_mean 

    # Correct Units 

    y_train_filtered_unnorm[:,0] = y_train_filtered_unnorm[:,0] * (X_train_filtered_unnorm[:,3] + X_train_filtered_unnorm[:,4])
    y_train_filtered_unnorm[:,1] *= y_train_filtered_unnorm[:,0]

    y_test_filtered_unnorm[:,0] = y_test_filtered_unnorm[:,0] * (X_test_filtered_unnorm[:,3] + X_test_filtered_unnorm[:,4])
    y_test_filtered_unnorm[:,1] *= y_test_filtered_unnorm[:,0]

    
    x_min, x_max = X_train_filtered_unnorm[:, feature_idx[0]].min() - 0.1, X_train_filtered_unnorm[:, feature_idx[0]].max() + 0.1
    y_min, y_max = X_train_filtered_unnorm[:, feature_idx[1]].min() - 0.1, X_train_filtered_unnorm[:, feature_idx[1]].max() + 0.1

    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 500),
                         np.linspace(y_min, y_max, 500))

    # Normalize grid
    xx_norm = (xx - train_mean[feature_idx[0]]) / train_std[feature_idx[0]]
    yy_norm = (yy - train_mean[feature_idx[1]]) / train_std[feature_idx[1]]

    # Step 3: Build input space
    X_vis = torch.zeros((xx_norm.ravel().shape[0], X_train.shape[1]), dtype=torch.float32)
    X_vis[:, feature_idx[0]] = torch.tensor(xx_norm.ravel(), dtype=torch.float32)
    X_vis[:, feature_idx[1]] = torch.tensor(yy_norm.ravel(), dtype=torch.float32)
    
    for k, v in fixed_values_norm.items():
        X_vis[:, k] = v

    # Step 4: Model predictions
    X_vis = X_vis.to(device)
    model.eval()
    with torch.no_grad():
        preds = model(X_vis).detach().cpu().numpy()   # shape (Ngrid, 2)

    # Changing the training and testing data to be M1f and M2f 
        
    y_train_filtered_corrected = np.empty((len(y_train_filtered_unnorm[:,0]), 2))
    y_test_filtered_corrected = np.empty((len(y_test_filtered_unnorm[:,0]), 2))
    
    y_train_filtered_corrected[:,0] = y_train_filtered_unnorm[:, 1]
    y_train_filtered_corrected[:,1] = y_train_filtered_unnorm[:, 0] - y_train_filtered_unnorm[:, 1]

    y_test_filtered_corrected[:,0] = y_test_filtered_unnorm[:, 1]
    y_test_filtered_corrected[:,1] = y_test_filtered_unnorm[:, 0] - y_test_filtered_unnorm[:, 1]

    # Now put in terms of Mass 1 and Mass 2 Final 
    # M_tot_f = (preds[:, 0]  * y_train_std[0] ) + y_train_mean[0]    # model predicts total final mass
    # M1_f = (preds[:, 1]  * y_train_std[1] ) + y_train_mean[1]      # model predicts final M1
    # M2_f = M_tot_f - M1_f
    print(preds[:, 0], y_train_std[0], y_train_mean[0], fixed_values[3] + fixed_values[4])
    M_tot_f = (preds[:, 0]  * y_train_std[0] ) + y_train_mean[0]    # model predicts total final mass / total initial mass
    M_tot_f *= (fixed_values[3] + fixed_values[4])
    M1_f = (preds[:, 1]  * y_train_std[1] ) + y_train_mean[1]      # model predicts final M1 / total final mass
    M1_f *= M_tot_f
    M2_f = M_tot_f - M1_f
    print(M_tot_f)
    # Reshape into grid for each output dimension
    M1_f = M1_f.reshape(xx.shape)
    M2_f = M2_f.reshape(xx.shape)

    # Step 5: Plot two panels
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    vmin = min(y_train_filtered_corrected[:,0].min(), y_train_filtered_corrected[:,1].min(),
           y_test_filtered_corrected[:,0].min(), y_test_filtered_corrected[:,1].min(),
           M1_f.min(), M2_f.min())
    vmax = max(y_train_filtered_corrected[:,0].max(), y_train_filtered_corrected[:,1].max(),
           y_test_filtered_corrected[:,0].max(), y_test_filtered_corrected[:,1].max(),
           M1_f.max(), M2_f.max())
    # print(y_train_filtered_corrected[:,0].min(), y_train_filtered_corrected[:,1].min(),
    #        y_test_filtered_corrected[:,0].min(), y_test_filtered_corrected[:,1].min(),
    #        M1_f.min(), M2_f.min())
    # print(y_train_filtered_corrected[:,0].max(), y_train_filtered_corrected[:,1].max(),
    #        y_test_filtered_corrected[:,0].max(), y_test_filtered_corrected[:,1].max(),
    #        M1_f.max(), M2_f.max())
    
    # print(vmin,vmax)
    
    cmap = plt.cm.coolwarm
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

    for i, (Z, ax, title) in enumerate(zip([M1_f, M2_f], axes, ['Star 1 Final Mass', 'Star 2 Final Mass'])):
        im = ax.contourf(xx, yy, Z, levels = 100, cmap=cmap, norm = norm)
        scatter1 = ax.scatter(X_train_filtered_unnorm[:, feature_idx[0]],
                              X_train_filtered_unnorm[:, feature_idx[1]],
                              c=y_train_filtered_corrected[:, i], cmap=cmap, norm = norm, edgecolor="k", marker="o", label="Train")
        scatter2 = ax.scatter(X_test_filtered_unnorm[:, feature_idx[0]],
                              X_test_filtered_unnorm[:, feature_idx[1]],
                              c=y_test_filtered_corrected[:, i], cmap=cmap, norm = norm, marker="^", label="Test")

        ax.set_xlabel(fr"{labels[0]}", fontsize=14)
        ax.set_ylabel(fr"{labels[1]}", fontsize=14)
        ax.tick_params(axis='both', which='major', labelsize=12)
        ax.set_title(title, fontsize=15)
    
   
    # After plotting the contours and scatters
    plt.tight_layout(rect=[0,0,0.9,1])  # leave 10% space on the right for the colorbar

    # Create the colorbar on a dedicated axis outside the panels
    cbar_ax = fig.add_axes([0.9999, 0.15, 0.02, 0.7])  # [left, bottom, width, height]
    cbar = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap),
                    cax=cbar_ax, orientation='vertical', shrink=0.8)
    cbar.set_label('Final Mass', fontsize=14)

    fig.suptitle(fr"$\mathrm{{M_1}} = {labels[2]}\ M_\odot,\ \mathrm{{M_2}} = {labels[3]}\ M_\odot,\ \mathrm{{Time}} = {round(10**(float(labels[4])),3)}\ \mathrm{{Gyr}}$", fontsize = 17)

    plt.tight_layout()
    return fig




In [ ]:
# 0:Age[Gyr],1:Rp,2:Vinf[km/s]3:Mass1[MSUN],4:Mass2[Msun],
from matplotlib.backends.backend_pdf import PdfPages
unique_times = np.unique(x_data_og[:,0])
unique_mass = np.unique(x_data_og[:,4])

with PdfPages('decision_boundary_plots_22f_regression_final.pdf') as pdf:
    for time in unique_times:
        for mass1 in unique_mass:
            for mass2 in unique_mass:
                # print("Time: ", str(time) + " Mass1: " + str(mass1) + " Mass2: " + str(mass2) + '\n' )
                labels = [r'$\rm r_p \, [R_{\odot}]$', r'$\rm Log_{10}(v_{inf} \, [km/s])$', str(mass1), str(round(mass2,2)), str(time)]
                
                fig = plot_4d_regression(model, train_dataset.data, train_dataset.labels, test_dataset.data, test_dataset.labels, train_mean, train_std, y_train_mean, y_train_std, 
                       feature_idx=(1, 2), fixed_values={0: time, 3: mass1, 4: mass2}, labels = labels)
                if fig != "No data points!":
                    # Save the current figure to the PDF
                    pdf.savefig(fig , bbox_inches='tight')  # Saves the current figure
                    plt.show()
                    plt.close(fig )    # Close it to avoid overlap

In [ ]:
# 0:Age[Gyr],1:Rp,2:Vinf[km/s]3:Mass1[MSUN],4:Mass2[Msun],
from matplotlib.backends.backend_pdf import PdfPages
unique_times = np.unique(x_data_og[:,0])
unique_mass = np.unique(x_data_og[:,4])
unique_times = np.array([-1])
unique_mass= np.array([8.0, 0.8])
for time in unique_times:
    for mass1 in unique_mass:
        for mass2 in unique_mass:
            # print("Time: ", str(time) + " Mass1: " + str(mass1) + " Mass2: " + str(mass2) + '\n' )
            labels = [r'$\rm r_p \, [R_{\odot}]$', r'$\rm Log_{10}(v_{inf} \, [km/s])$', str(mass1), str(round(mass2,2)), str(time)]
            
            fig = plot_4d_regression(model, train_dataset.data, train_dataset.labels, test_dataset.data, test_dataset.labels, train_mean, train_std, y_train_mean, y_train_std, 
                    feature_idx=(1, 2), fixed_values={0: time, 3: mass1, 4: mass2}, labels = labels)
            if fig != "No data points!":
                fig.show()

            print(fig)